# Interactive Agent Session: Chapter 3 — OpenAI Assistants Legal Compliance Auditor

**The Business Problem:** Legal teams manually review 100+ page vendor contracts. A single missed liability clause can cost millions in litigation.

**The Solution:** OpenAI Assistants API with **Stateful Threads** — the agent remembers the entire contract audit across multiple exchanges, audits multiple clauses, and flags risk levels.

**Key Concept:** Unlike stateless chat completions, Assistants store conversation history on OpenAI servers — enabling multi-turn audits without re-sending the full context.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet openai python-dotenv

In [ ]:
%%capture legal_logs

import os, time
from dotenv import load_dotenv
import openai

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
USE_SIMULATION = not api_key
if not USE_SIMULATION:
    client = openai.Client(api_key=api_key)

# ── CONTRACT CLAUSES TO AUDIT ──
CLAUSES = [
    {
        "id": "CL-14",
        "title": "Limitation of Liability",
        "text": "The Vendor shall not be responsible for any damages, errors, or losses including financial loss, even if caused by gross negligence or willful misconduct."
    },
    {
        "id": "CL-22",
        "title": "Data Handling & Privacy",
        "text": "The Vendor may share anonymized customer data with third-party analytics partners for the purpose of service improvement. No prior consent from the Client is required."
    },
    {
        "id": "CL-31",
        "title": "Termination & Exit",
        "text": "Upon termination, the Vendor retains the right to hold all Client data for up to 24 months. Data export fees of $50,000 apply for early retrieval."
    }
]

def audit_clause(clause):
    if USE_SIMULATION:
        sim = {
            "CL-14": {"risk": "CRITICAL", "finding": "This clause eliminates ALL liability including gross negligence. This is unconscionable in most jurisdictions. Recommend: Add a liability cap tied to contract value and exclude gross negligence/willful misconduct from the limitation."},
            "CL-22": {"risk": "HIGH", "finding": "Sharing customer data without consent violates GDPR Article 6 and CCPA Section 1798.100. Recommend: Require explicit opt-in consent and list all third-party partners in a data processing addendum."},
            "CL-31": {"risk": "MEDIUM", "finding": "24-month data retention post-termination is excessive. $50K exit fee creates vendor lock-in. Recommend: Reduce to 90 days retention, provide free data export in standard formats (CSV/JSON)."}
        }
        return sim.get(clause["id"], {"risk": "LOW", "finding": "No issues found."})
    
    asst = client.beta.assistants.create(
        name="Legal Compliance Auditor",
        instructions="You are a senior legal compliance officer specializing in vendor contract risk. For each clause: 1) Assign risk level (CRITICAL/HIGH/MEDIUM/LOW), 2) Cite specific legal issues, 3) Provide a concrete recommendation. Be concise.",
        model="gpt-4o-mini"
    )
    thread = client.beta.threads.create()
    client.beta.threads.messages.create(thread_id=thread.id, role="user", content=f"Audit this contract clause [{clause['id']}]: {clause['text']}")
    run = client.beta.threads.runs.create_and_poll(thread_id=thread.id, assistant_id=asst.id, max_completion_tokens=300)
    if run.status == "completed":
        msgs = client.beta.threads.messages.list(thread_id=thread.id)
        text = msgs.data[0].content[0].text.value
        risk = "CRITICAL" if "critical" in text.lower() else "HIGH" if "high" in text.lower() else "MEDIUM" if "medium" in text.lower() else "LOW"
        return {"risk": risk, "finding": text}
    return {"risk": "UNKNOWN", "finding": "Audit pending..."}

audit_results = []
for c in CLAUSES:
    result = audit_clause(c)
    audit_results.append({"clause": c, "audit": result})

In [ ]:
from IPython.display import display, HTML

risk_colors = {"CRITICAL": "#ef4444", "HIGH": "#f97316", "MEDIUM": "#eab308", "LOW": "#22c55e", "UNKNOWN": "#666"}
clauses_html = ""
for r in audit_results:
    c = r["clause"]
    a = r["audit"]
    color = risk_colors.get(a["risk"], "#666")
    clauses_html += '<div style="margin-bottom:25px; border-left:4px solid ' + color + '; padding-left:20px;">'
    clauses_html += '<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">'
    clauses_html += '<span style="font-size:11px; font-weight:700; color:' + color + '; background:' + color + '22; padding:2px 10px; border-radius:50px;">' + a["risk"] + '</span>'
    clauses_html += '<span style="font-size:12px; color:#94a3b8;">' + c["id"] + ' — ' + c["title"] + '</span>'
    clauses_html += '</div>'
    clauses_html += '<div style="font-size:12px; color:#64748b; font-style:italic; margin-bottom:8px; padding:10px; background:#0f172a; border-radius:8px;">' + c["text"] + '</div>'
    clauses_html += '<div style="font-size:13px; color:#e2e8f0; line-height:1.6;">' + a["finding"] + '</div>'
    clauses_html += '</div>'

html = '<style>@import url("https://fonts.googleapis.com/css2?family=Inter:wght@400;700&family=Unbounded:wght@800&display=swap");</style>'
html += '<div style="max-width:900px; margin:30px auto; font-family:Inter,sans-serif; background:#020617; padding:50px; border-radius:4px; border-left:15px solid #1e293b; box-shadow:0 30px 80px rgba(0,0,0,0.5); color:#fff;">'
html += '<div style="font-family:Unbounded,sans-serif; font-size:14px; color:#64748b; letter-spacing:2px; margin-bottom:10px;">OPENAI_ASSISTANTS_API</div>'
html += '<div style="font-size:10px; color:#1e293b; font-weight:800; letter-spacing:2px; border:1px solid #1e293b; padding:2px 10px; display:inline-block; margin-bottom:30px; color:#94a3b8;">OFFICIAL AUDIT REPORT</div>'
html += clauses_html
html += '</div>'
display(HTML(html))